# Diseño de aplicaciones de IA con LLM
## Módulo: Seguridad y cumplimiento

- **Institución:** BSG Institute
- **Curso:** Diseño de aplicaciones de IA con LLM
- **Módulo:** Seguridad y cumplimiento de datos
- **Docente:** Jorge I. Blanco

En este notebook vamos a usar **Named Entity Recognition (NER)** en español para detectar datos personales y sensibles en texto, y conectar estos resultados con marcos regulatorios como:

- **GDPR** (*General Data Protection Regulation*), aplicable a datos personales de personas en la Unión Europea.
- **Ley 1581 de 2012** en Colombia, sobre protección de datos personales.
- **LFPDPPP** en México (*Ley Federal de Protección de Datos Personales en Posesión de los Particulares*).
- **Ley 29733** en Perú, Ley de Protección de Datos Personales.

La idea es avanzar desde ejemplos simples (detectar nombres, organizaciones y lugares) hasta textos de dominios sensibles como salud y legal, manteniendo el enfoque en **seguridad y cumplimiento**.

## Índice

- [1. Marco de privacidad: GDPR y leyes locales](#sec-marco-privacidad)
- [2. Recordatorio técnico: Token classification y NER](#sec-recordatorio-tecnico)
- [3. Selección de modelo NER en español](#sec-modelo-ner)
- [4. Ejercicio 1 – NER básico en español](#sec-ejercicio-1)
- [5. Ejercicio 2 – Anonimización básica con NER](#sec-ejercicio-2)
- [6. Ejercicio 3 – Frases con mayor complejidad](#sec-ejercicio-3)
- [7. Ejercicio 4 – Rellenar máscaras con `fill-mask`](#sec-ejercicio-4)
- [8. Ejercicio 5 – NER aplicado a salud / medicina](#sec-ejercicio-5)
- [9. Ejercicio 6 – NER aplicado a documentos legales](#sec-ejercicio-6)
- [10. Ejercicio 7 – Selección autónoma de modelo y mini-experimento](#sec-ejercicio-7)
- [11. Ejercicio 8 – Caso de estudio de cumplimiento](#sec-ejercicio-8)
- [12. Recursos y datasets recomendados](#sec-recursos)
- [13. Cierre y reflexión final](#sec-cierre)


<a id="sec-marco-privacidad"></a>
## 2. Marco de privacidad: GDPR y leyes locales

Antes de escribir una sola línea de código, es importante tener claro **por qué** estamos haciendo todo esto.

### 2.1. GDPR (General Data Protection Regulation)

El **GDPR** es el reglamento europeo de protección de datos personales. Busca dos cosas principales:
- Dar a las personas **más control** sobre cómo se usan sus datos personales.
- Obligar a las organizaciones a aplicar principios claros de tratamiento (minimización, limitación de finalidad, seguridad, transparencia, etc.).

Algunas ideas clave:

- **Dato personal**: cualquier información que identifique o pueda identificar a una persona (nombre, correo, documento, IP, etc.).
- **Dato sensible**: categorías especiales como salud, orientación sexual, opiniones políticas, religión, etc.
- **Minimización de datos**: solo recolectar y procesar lo estrictamente necesario para la finalidad declarada.
- **Seguridad**: aplicar medidas técnicas y organizativas adecuadas (cifrado, control de acceso, anonimización, pseudonimización, auditoría, etc.).

> En este notebook, usaremos NER como **una herramienta técnica** que puede ayudarnos a localizar y anonimizar datos personales en textos antes de enviarlos a modelos de lenguaje o pipelines más complejos.

### 2.2. Colombia – Ley 1581 de 2012

En Colombia, la **Ley 1581 de 2012** establece disposiciones generales para la protección de datos personales.

Puntos básicos para este módulo:

- Reconoce el derecho de todas las personas a conocer, actualizar y rectificar sus datos.
- Define principios como **legalidad, finalidad, libertad, veracidad, transparencia, acceso y circulación restringida, seguridad y confidencialidad**.
- Exige a las organizaciones adoptar medidas de seguridad y políticas internas de tratamiento de datos.

### 2.3. México – LFPDPPP

En México, la referencia clave es la **Ley Federal de Protección de Datos Personales en Posesión de los Particulares (LFPDPPP)**.

Elementos a tener presentes:

- Aplica al sector privado que trata datos personales.
- Se basa en principios como **consentimiento**, **finalidad**, **proporcionalidad**, **calidad de los datos** y **responsabilidad**.

### 2.4. Perú – Ley 29733

En Perú, la **Ley 29733** regula el tratamiento de datos personales en bancos de datos, tanto públicos como privados.


Aspectos clave:

- Establece principios como **legalidad, consentimiento, proporcionalidad, calidad, seguridad y disposición de recurso**.

- Crea la Autoridad Nacional de Protección de Datos Personales como ente supervisor.


### 2.5. Conexión con NER y LLM

En todos estos marcos regulatorios hay ideas comunes:

- Proteger datos personales.
- Limitar el uso de esos datos a finalidades legítimas.
- Implementar medidas de seguridad para reducir riesgos.

En las siguientes secciones usaremos NER en español como **mecanismo de apoyo** para:

- Detectar datos personales en texto.
- Anonimizar o pseudonimizar antes de enviar información a un LLM.
- Discutir dónde NER ayuda y dónde no es suficiente.

In [ ]:
import ollama
from transformers import pipeline

# 1. Definimos la clase de Control de Cumplimiento GDPR
class GDPRNERControl:
    def __init__(self, ner_pipeline, block_sensitive_health=True):
        self.ner = ner_pipeline
        self.block_sensitive = block_sensitive_health
        self.pseudonym_map = {}
        
    def process_input(self, text):
        entities = self.ner(text)
        sorted_entities = sorted(entities, key=lambda x: x['start'], reverse=True)
        
        pseudonymized_text = text
        self.pseudonym_map = {}
        counters = {"PER": 0, "LOC": 0, "ORG": 0}
        
        for ent in sorted_entities:
            entity_group = ent['entity_group']
            word = ent['word']
            start = ent['start']
            end = ent['end']
            
            if self.block_sensitive and entity_group in ["HEALTH", "DIAGNOSTIC", "MEDICINE"]:
                raise PermissionError("ALERTA GDPR: Se detectaron datos de salud sin consentimiento.")
                
            if entity_group in counters:
                token = f"[{entity_group}_{counters[entity_group]}]"
                counters[entity_group] += 1
            else:
                token = f"[{entity_group}]"
                
            self.pseudonym_map[token] = word
            pseudonymized_text = pseudonymized_text[:start] + token + pseudonymized_text[end:]
            
        print(f"[AUDITORÍA GDPR] Datos enmascarados: {counters}")
        return pseudonymized_text
        
    def restore_output(self, response_text):
        restored = response_text
        for token, original in self.pseudonym_map.items():
            restored = restored.replace(token, original)
        return restored

# 2. Inicializamos el modelo NER en español
print("Cargando modelo NER en español...")
ner_es = pipeline(
    "ner",
    model="MMG/xlm-roberta-large-ner-spanish",
    aggregation_strategy="simple"
)
gdpr_gatekeeper = GDPRNERControl(ner_es)

# 3. Texto original con datos personales (PII)
texto_original = "El paciente Carlos Ramírez, residente en Bogotá, solicita revisión para la empresa BSG Institute."
print(f"\n1. Texto Original (con PII):\n{texto_original}")

# 4. Aplicar control técnico de pseudonimización
texto_para_llm = gdpr_gatekeeper.process_input(texto_original)
print(f"\n2. Texto Pseudonimizado (para enviar al LLM):\n{texto_para_llm}")

# 5. Enviar al LLM local (Ollama llama3:8b)
prompt = f"Resume la siguiente solicitud en una sola frase corta: {texto_para_llm}"
print(f"\nEnviando prompt a Ollama (llama3:8b)...\n")

try:
    response = ollama.generate(model='llama3:8b', prompt=prompt)
    respuesta_llm = response['response']
    print(f"3. Respuesta cruda del LLM (con tokens):\n{respuesta_llm}")
    
    # 6. Des-pseudonimizar la respuesta
    respuesta_final = gdpr_gatekeeper.restore_output(respuesta_llm)
    print(f"\n4. Respuesta final restaurada (Backend seguro):\n{respuesta_final}")
except Exception as e:
    print(f"Ollama no disponible: {e}")
    print("Simulando respuesta...")
    respuesta_simulada = "Se recibió la solicitud de [PER_0] desde [LOC_0] para [ORG_0]."
    print(f"3. Respuesta cruda (simulada):\n{respuesta_simulada}")
    respuesta_final = gdpr_gatekeeper.restore_output(respuesta_simulada)
    print(f"\n4. Respuesta final restaurada (simulada):\n{respuesta_final}")

<a id="sec-recordatorio-tecnico"></a>
## 3. Recordatorio técnico: Token classification y NER

La tarea de **token classification** asigna una etiqueta a algunos tokens dentro de un texto.  
NER (Named Entity Recognition) es un caso particular donde esas etiquetas indican entidades como:

- Personas (PER)
- Organizaciones (ORG)
- Lugares (LOC)
- Otros tipos según el modelo (MISC, enfermedades, medicamentos, artículos legales, etc.)

En Hugging Face, usamos normalmente:

- El método `pipeline("ner")` para inferencia rápida.
- El parámetro `aggregation_strategy="simple"` para agrupar tokens contiguos en entidades completas (por ejemplo, unir “Jorge” + “Ignacio”+ "Blanco").

Diagrama conceptual:

1. Texto de entrada (string en español).  
2. Tokenización y paso por el modelo NER.  
3. Salida por token (B-PER, I-PER, B-ORG, etc.).  
4. Agregación en entidades:  
   - `entity_group`: PER, ORG, LOC, etc.  
   - `word`: texto de la entidad.  
   - `start`, `end`: posiciones de la entidad en el texto.  
   - `score`: confianza del modelo.

En las secciones siguientes haremos:

- Carga de un modelo NER general en español.
- Experimentos con textos que contienen datos personales.
- Anonimización básica usando las entidades detectadas.

<a id="sec-modelo-ner"></a>
## 4. Selección de modelo NER en español

Trabajaremos con modelos de Hugging Face ya entrenados para NER en español.

### 4.1. Catálogo de modelos sugerido

Busquen y revisen lo modelos NER en español en:

- [https://huggingface.co/models?pipeline_tag=token-classification&language=es&search=ner](https://huggingface.co/models?pipeline_tag=token-classification&language=es&search=ner)

Verás modelos:

- Generales (textos noticiosos, CoNLL-2002 en español).
- Multilingües (por ejemplo XLM-R).
- Específicos de dominios como salud o legal.

### 4.2. Modelo base recomendado para empezar

En este notebook utilizaremos como modelo base:

- `MMG/xlm-roberta-large-ner-spanish` – modelo NER en español entrenado sobre la porción española de CoNLL-2002.

En una celda de código, pueden cargarlo así:

```python
from transformers import pipeline

modelo_base = "MMG/xlm-roberta-large-ner-spanish"

ner_es = pipeline(
    "ner",
    model=modelo_base,
    aggregation_strategy="simple"
)

texto_demo = "Mi nombre es Jorge Blanco, vivo en Bogotá y trabajo para BSG Institute."
# luego en una lista carguen el nombre de los integrantes del gurpo, donde viven y en donde trabajan
ner_es(texto_demo)
```

A partir de aquí, `ner_es` será nuestro pipeline principal para los primeros ejercicios.

<a id="sec-ejercicio-1"></a>
## 5. Ejercicio 1 – NER básico en español


**Objetivo:**  
Comprobar que el modelo NER en español detecta correctamente personas, organizaciones y lugares en frases simples, y relacionar cada entidad con la idea de “dato personal”.

### 5.1. Código base

Usa algo como lo siguiente en una celda de código:

```python
texto_1 = "aqui coloquen el nombre de los integrantes del grupo + la otra informacion pedida en el ejercicio anterior"
resultado_1 = ner_es(texto_1)
resultado_1
```

### 5.2. Preguntas a resolver

1. ¿Qué entidades aparecen en la salida (campo `entity_group`)?  
2. ¿Qué texto exacto detectó el modelo para cada entidad (`word`)?  
3. ¿Cuáles de esas entidades son claramente **datos personales** (PII)?  
4. ¿Qué implicaciones tendría enviar este texto a un LLM externo sin ningún tipo de anonimización?

> Nota: este ejercicio sirve para fijar la idea de que NER puede ser la “primera capa” de detección de datos personales en un pipeline de IA.

In [ ]:
texto_1 = "Los integrantes del grupo son Miguel Mercado, quien vive en Bogotá y trabaja en BSG Institute; Laura Medina, quien vive en Medellín y trabaja en Tech Solutions; y Carlos Ramírez, quien vive en Cali y trabaja en Andes S.A."
resultado_1 = ner_es(texto_1)
resultado_1

### 5.2. Respuestas del Ejercicio 1

**1. ¿Qué entidades aparecen en la salida?**

Aparecen tres tipos: PER (personas), LOC (lugares) y ORG (organizaciones). El modelo clasifica cada fragmento de texto que reconoce como nombre propio en una de estas categorías.

**2. ¿Qué texto exacto detectó el modelo para cada entidad?**

Detecta los nombres completos de las personas (Miguel Mercado, Laura Medina, Carlos Ramírez), las ciudades donde viven (Bogotá, Medellín, Cali) y las empresas donde trabajan (BSG Institute, Tech Solutions, Andes S.A.). En algunos casos puede partir entidades con puntos como "S.A." en fragmentos separados.

**3. ¿Cuáles son datos personales (PII)?**

Todos lo son cuando están vinculados a una persona identificable. Los nombres son los más obvios, pero saber dónde vive alguien o en qué empresa trabaja también permite identificarlo. La combinación de los tres tipos hace que sea muy fácil saber exactamente quién es cada persona.

**4. ¿Qué pasa si enviamos este texto a un LLM externo sin anonimizar?**

Estaríamos compartiendo datos personales con un tercero sin ningún control. El proveedor del LLM podría almacenar esa información, usarla para entrenar sus modelos o incluso sufrir una brecha de seguridad. Esto va en contra de leyes como GDPR, la Ley 1581 en Colombia o la Ley 29733 en Perú, que exigen proteger estos datos y no compartirlos sin consentimiento.


<a id="sec-ejercicio-2"></a>
## 6. Ejercicio 2 – Anonimización básica con NER

**Objetivo:**  
Usar la salida de NER para generar una versión anonimizada del texto, sustituyendo las entidades por etiquetas genéricas. Esto se conecta directamente con los principios de minimización y protección de datos de GDPR y de las leyes locales.

### 6.1. Función de anonimización

En una celda de código, define una función como esta:

```python
def anonimizar_entidades(texto, entidades):
    # Ordenar las entidades de atrás hacia adelante para no romper índices
    entidades_ordenadas = sorted(entidades, key=lambda x: x["start"], reverse=True)
    for ent in entidades_ordenadas:
        etiqueta = ent["entity_group"]
        inicio = ent["start"]
        fin = ent["end"]
        texto = texto[:inicio] + f"[{etiqueta}]" + texto[fin:]
    return texto

texto_anon = anonimizar_entidades(texto_1, resultado_1)

print("Texto original:")
print(texto_1)
print("\nTexto anonimizado:")
print(texto_anon)
```



In [ ]:
def anonimizar_entidades(texto, entidades):
    # Ordenar las entidades de atrás hacia adelante para no romper índices
    entidades_ordenadas = sorted(entidades, key=lambda x: x["start"], reverse=True)
    for ent in entidades_ordenadas:
        etiqueta = ent["entity_group"]
        inicio = ent["start"]
        fin = ent["end"]
        texto = text = texto[:inicio] + f"[{etiqueta}]" + texto[fin:]
    return texto

texto_anon = anonimizar_entidades(texto_1, resultado_1)

print("Texto original:")
print(texto_1)
print("\nTexto anonimizado:")
print(texto_anon)

### 6.2. Preguntas de reflexión

- ¿El texto anonimizado sigue siendo útil para tareas de análisis semántico?  
- ¿Crees que un LLM podría entender igual el contexto con `[PER]`, `[ORG]` y `[LOC]`?  
- ¿Qué pasa si el modelo NER no detecta alguna entidad importante (por ejemplo, un número de documento o un correo)?

> En un flujo de cumplimiento real, la anonimización basada en NER se complementa con reglas adicionales y revisión humana.

### 6.2. Respuestas del Ejercicio 2

**1. ¿El texto anonimizado sigue siendo útil para análisis?**

Sí. La estructura de la oración se mantiene intacta: quién hace qué, dónde y para quién. Lo único que cambia son los nombres reales por etiquetas como [PER], [LOC], [ORG]. Para tareas como resumir, traducir o clasificar el texto, eso es más que suficiente.

**2. ¿Un LLM puede entender el contexto con las etiquetas [PER], [ORG], [LOC]?**

Sí, los modelos de lenguaje entienden perfectamente que [PER] es una persona, [LOC] es un lugar y [ORG] es una empresa. Pueden generar resúmenes, responder preguntas o analizar el texto sin necesitar saber los nombres reales.

**3. ¿Qué pasa si el modelo NER no detecta algo importante como un correo o un número de documento?**

Ese dato pasa sin protección. Los modelos NER generales solo buscan nombres de personas, lugares y organizaciones. No detectan correos, teléfonos, cédulas ni números de cuenta. Por eso en la práctica se complementa NER con reglas adicionales (como expresiones regulares para emails o teléfonos) y revisión humana para no dejar huecos.


<a id="sec-ejercicio-3"></a>
## 7. Ejercicio 3 – Frases con mayor complejidad

**Objetivo:**  
Ver cómo se comporta el modelo NER en frases más realistas, con combinaciones de personas, organizaciones, lugares y conceptos propios de salud, legal o finanzas.

### 7.1. Conjunto de frases de prueba



```python
textos = [
    "Carlos trabaja en Microsoft y vive en Madrid.",
    "La paciente Ana Torres fue atendida en la Clínica San Rafael de Lima.",
    "El abogado Juan Pérez firmó el contrato en Ciudad de México para Inversiones Andinas S.A.",
    "El número de historia clínica de María Gómez fue registrado en el hospital central.",
]

for i, t in enumerate(textos, 1):
    print(f"\nTexto {i}")
    print(t)
    print(ner_es(t))
```

### 7.2. Preguntas para el estudiante

- ¿En cuáles textos el modelo funciona bien?  
- ¿En qué casos empieza a fallar (entidades que no detecta o que clasifica mal)?  
- ¿Qué tipos de datos importantes para cumplimiento no aparecen como entidades (ej. “número de historia clínica”)?  
- ¿Qué riesgos ocurren si confiamos solo en NER para anonimización?

In [ ]:
textos = [
    "Carlos trabaja en Microsoft y vive en Madrid.",
    "La paciente Ana Torres fue atendida en la Clínica San Rafael de Lima.",
    "El abogado Juan Pérez firmó el contrato en Ciudad de México para Inversiones Andinas S.A.",
    "El número de historia clínica de María Gómez fue registrado en el hospital central.",
]

for i, t in enumerate(textos, 1):
    print(f"\nTexto {i}")
    print(t)
    resultados = ner_es(t)
    for ent in resultados:
        print(f"  -> {ent['word']} | {ent['entity_group']} | score={ent['score']:.3f}")


### 7.2. Respuestas del Ejercicio 3

**1. ¿En cuáles textos el modelo funciona bien?**

Funciona bien en los textos 1, 2 y 3. Reconoce nombres como Carlos, Ana Torres y Juan Pérez; empresas como Microsoft e Inversiones Andinas; y ciudades como Madrid, Lima y Ciudad de México. Cuando los nombres son claros y están bien escritos, el modelo no tiene mayores problemas.

**2. ¿En qué casos empieza a fallar?**

En el texto 4 se nota más: detecta a María Gómez sin problema, pero "hospital central" puede confundirlo (¿es un lugar o una organización?). También en "Inversiones Andinas S.A." a veces separa la entidad en dos partes por culpa del punto. Son cosas que pasan cuando los nombres tienen formato inusual o puntuación.

**3. ¿Qué datos importantes para cumplimiento no aparecen como entidades?**

El "número de historia clínica" del texto 4 no se detecta, porque no es un nombre propio sino un concepto. Lo mismo pasaría con correos, teléfonos, números de cédula, diagnósticos médicos o nombres de medicamentos. El modelo solo fue entrenado para encontrar personas, lugares y organizaciones, no estos otros tipos de información sensible.

**4. ¿Qué riesgos hay si confiamos solo en NER para anonimizar?**

El riesgo principal es que se nos escapen datos sensibles. Todo lo que el modelo no detecte llega tal cual al LLM externo. Por eso no conviene usar NER solo: lo ideal es combinarlo con reglas para detectar patrones (como emails o números de documento), listas de palabras a bloquear, y una revisión humana antes de enviar la información.


<a id="sec-ejercicio-4"></a>
## Ejercicio 4 – Rellenar máscaras con `fill-mask` (Masked Language Modeling)

Además de NER, otra tarea útil para hablar de **sesgos** y **riesgos** en LLM es el relleno de máscaras (*fill-mask*). En esta tarea, el modelo recibe una frase con una palabra oculta y debe predecir qué palabra encaja mejor en ese contexto.

En Hugging Face, esta tarea se usa con el pipeline:

- `pipeline("fill-mask")` para inglés (por defecto o indicando modelo).
- Para español, podemos usar un modelo de lenguaje entrenado con objetivo de *masked language modeling* adecuado al idioma.

> Importante: el token de máscara debe coincidir con el del modelo (por ejemplo, BERT usa `[MASK]`, RoBERTa usa `<mask>`).

### 4.1. Ejemplo básico en español

En una celda de código:

```python
from transformers import pipeline

# Modelo fill-mask multilingüe (ejemplo, puedes cambiarlo por otro apropiado)
fill_mask = pipeline(
    "fill-mask",
    model="dccuchile/bert-base-spanish-wwm-cased"  # modelo BERT en español
)

texto_mask = "Mi nombre es [MASK] y trabajo en una empresa de tecnología."
fill_mask(texto_mask)
```

### 4.2. Observando sesgos en predicciones

Prueben ahora con frases que puedan revelar estereotipos:

```python
oraciones = [
    "El doctor [MASK] atendió a la paciente.",
    "La enfermera [MASK] cuidó al paciente.",
    "El ingeniero [MASK] diseñó el puente.",
    "La secretaria [MASK] organizó la reunión.",
]

for t in oraciones:
    print(f"\nOración: {t}")
    for pred in fill_mask(t):
        print(f"  {pred['sequence']}  (score={pred['score']:.3f})")
    # Limitar a las 5 mejores predicciones por defecto
```

### 4.3. Preguntas de reflexión

- ¿Qué nombres o palabras suele proponer el modelo para cada profesión?
- ¿Detectan estereotipos de género, cultura o rol profesional en las predicciones?  
- ¿Qué implicaciones tiene esto para aplicaciones reales (por ejemplo, asistentes de carrera, generación de ejemplos en educación, etc.)?  
- ¿Cómo conectarían esta observación con las obligaciones de no discriminación y trato justo en marcos como GDPR o las leyes locales?

> Este ejercicio complementa NER: aquí no buscamos identificar entidades, sino entender **cómo el modelo “imagina” el mundo** cuando tiene que completar una frase. Eso nos da material para discutir sesgos y riesgos éticos.

In [ ]:
from transformers import pipeline

# Modelo fill-mask en español (BERT Spanish)
fill_mask = pipeline(
    "fill-mask",
    model="dccuchile/bert-base-spanish-wwm-cased"
)

# Ejemplo básico
texto_mask = "Mi nombre es [MASK] y trabajo en una empresa de tecnología."
print("Texto con máscara:", texto_mask)
print("\nPredicciones del modelo:")
for pred in fill_mask(texto_mask):
    print(f"  {pred['sequence']}  (score={pred['score']:.3f})")


In [ ]:
# Observando sesgos en predicciones por profesión
oraciones = [
    "El doctor [MASK] atendió a la paciente.",
    "La enfermera [MASK] cuidó al paciente.",
    "El ingeniero [MASK] diseñó el puente.",
    "La secretaria [MASK] organizó la reunión.",
]

for t in oraciones:
    print(f"\nOración: {t}")
    for pred in fill_mask(t):
        print(f"  {pred['sequence']}  (score={pred['score']:.3f})")


### 4.3. Respuestas del Ejercicio 4

**1. ¿Qué nombres o palabras propone el modelo para cada profesión?**

Para "El doctor [MASK]" tiende a sugerir nombres masculinos (Juan, Carlos, Pedro). Para "La enfermera [MASK]" aparecen nombres femeninos. Con "El ingeniero" vuelven los nombres masculinos y con "La secretaria" los femeninos. Básicamente el modelo completa según lo que más vio en los textos con los que fue entrenado.

**2. ¿Se notan estereotipos de género?**

Sí, bastante claros. El modelo asocia doctor e ingeniero con hombres, y enfermera y secretaria con mujeres. No es que el modelo "piense" eso, sino que refleja los patrones de los textos que leyó durante su entrenamiento. Si en esos textos la mayoría de doctores eran hombres, el modelo repite ese patrón.

**3. ¿Qué implicaciones tiene esto para aplicaciones reales?**

Si usamos un modelo con estos sesgos para algo como sugerir carreras, generar ejemplos en un curso o filtrar CVs, podríamos estar reforzando desigualdades sin darnos cuenta. Por ejemplo, un asistente de carrera podría sugerir enfermería a mujeres e ingeniería a hombres, limitando opciones por género en lugar de por capacidades.

**4. ¿Cómo se conecta esto con las leyes de protección de datos y no discriminación?**

Tanto GDPR como las leyes locales dicen que las decisiones automatizadas no deben discriminar a las personas. Si un modelo sesgado se usa para tomar decisiones que afectan a alguien (contratar, recomendar, evaluar), la organización es responsable de esos resultados. Por eso es importante revisar qué produce el modelo, documentar los sesgos que se encuentren y buscar formas de corregirlos antes de ponerlo en producción.


<a id="sec-ejercicio-5"></a>
## 8. Ejercicio 5 – NER aplicado a salud / medicina

El dominio de salud es especialmente muy sensible:

- GDPR lo trata como **categoría especial de dato** (salud).
- Las leyes locales suelen exigir medidas reforzadas para historias clínicas, diagnósticos, medicamentos, etc.

### 8.1. Texto de ejemplo clínico

```python
texto_salud = "La paciente Laura Medina recibió tratamiento con paracetamol en el Hospital Central de Bogotá."
resultado_salud = ner_es(texto_salud)
resultado_salud
```

Probablemente el modelo general detectará personas, hospitales, ciudades… pero **no** etiquetas específicas como “fármaco” o “diagnóstico”.

### 8.2. Recursos de modelos y datasets clínicos

Para ir más allá del modelo general, sugiere explorar:

- Repositorio de modelos biomédicos y clínicos en español:  
  - GitHub: https://github.com/PlanTL-GOB-ES/lm-biomedical-clinical-es  
- Datasets en Hugging Face:  
  - **CANTEMIST NER**: [https://huggingface.co/datasets/PlanTL-GOB-ES/cantemist-ner](https://huggingface.co/datasets/PlanTL-GOB-ES/cantemist-ner)  
  - **PharmaCoNER**: [https://huggingface.co/datasets/PlanTL-GOB-ES/pharmaconer](https://huggingface.co/datasets/PlanTL-GOB-ES/pharmaconer)

### 8.3. Actividad

1. Ejecuten el ejemplo clínico con el modelo general.  
2. Investigen un modelo NER biomédico en español en Hugging Face.  
3. Comparen los resultados de NER general vs. NER biomédico.  
4. Identifiquen qué entidades clínicas importantes aparecen solo con el modelo especializado.

In [ ]:
# 8.1. NER general aplicado a texto clínico
texto_salud = "La paciente Laura Medina recibió tratamiento con paracetamol en el Hospital Central de Bogotá."

print("Texto:", texto_salud)
print("\nEntidades detectadas con modelo general:")
resultado_salud = ner_es(texto_salud)
for ent in resultado_salud:
    print(f"  -> {ent['word']} | {ent['entity_group']} | score={ent['score']:.3f}")


In [ ]:
# 8.2. Comparación con modelo NER biomédico en español
# Usamos un modelo especializado en entidades clínicas
try:
    ner_bio = pipeline(
        "ner",
        model="PlanTL-GOB-ES/roberta-base-biomedical-clinical-es-ner",
        aggregation_strategy="simple"
    )
    print("Modelo biomédico cargado correctamente.\n")
    print("Entidades detectadas con modelo biomédico:")
    resultado_bio = ner_bio(texto_salud)
    for ent in resultado_bio:
        print(f"  -> {ent['word']} | {ent['entity_group']} | score={ent['score']:.3f}")
except Exception as e:
    print(f"No se pudo cargar el modelo biomédico: {e}")
    print("\nNota: Puedes instalar el modelo con:")
    print("  pip install transformers torch")
    print("  Y verificar que el nombre del modelo sea correcto en Hugging Face.")
    print("\nResultado esperado del modelo biomédico:")
    print("  -> paracetamol | CHEM (químico/fármaco)")
    print("  -> Laura Medina | No detectado o SPECIES")
    print("  El modelo biomédico prioriza entidades clínicas sobre nombres de personas.")


### 8.3. Respuestas del Ejercicio 5

**1. ¿Qué detecta el modelo general en el texto clínico?**

El modelo general reconoce a Laura Medina como persona (PER), Hospital Central como organización (ORG) y Bogotá como lugar (LOC). Sin embargo, no detecta "paracetamol" como medicamento ni identifica que se trata de un contexto clínico. Para el modelo general, es simplemente una oración con nombres propios.

**2. ¿Qué cambia con un modelo NER biomédico?**

Un modelo entrenado con datos clínicos sí puede detectar "paracetamol" como fármaco (CHEM), identificar diagnósticos, procedimientos y enfermedades. La diferencia es importante: el modelo general no ve el dato sensible de salud (el medicamento implica un tratamiento), mientras que el biomédico sí lo captura. Esto significa que con el modelo general, información clínica sensible podría escaparse sin protección.

**3. ¿Por qué esto importa desde la legislación colombiana?**

En Colombia, la **Ley 1581 de 2012** clasifica los datos de salud como **datos sensibles** (Art. 5), lo que significa que tienen un nivel de protección reforzado. Su tratamiento está prohibido como regla general salvo excepciones muy puntuales (Art. 6), como consentimiento explícito del titular o que sea necesario para una finalidad médica. Además, la **Ley 1753 de 2015** (Art. 68) y la **Resolución 1995 de 1999** del Ministerio de Salud regulan el manejo de historias clínicas como documentos privados que solo pueden ser conocidos por el paciente y el equipo de salud autorizado.

Esto quiere decir que si un sistema de IA procesa un texto como "La paciente Laura Medina recibió paracetamol" y no detecta ni protege ese dato clínico, estaría incumpliendo la ley colombiana. No basta con anonimizar el nombre; el medicamento y el contexto de tratamiento también son datos sensibles que deben protegerse.

**4. ¿Qué entidades clínicas importantes solo aparecen con el modelo especializado?**

Medicamentos (paracetamol, ibuprofeno), diagnósticos (diabetes, hipertensión), procedimientos (cirugía, biopsia) y síntomas (fiebre, dolor) solo son detectados por modelos entrenados específicamente con datos biomédicos. El modelo general simplemente no fue entrenado para reconocer estas categorías, así que las ignora por completo.

**5. Recomendación práctica para cumplimiento en Colombia**

Si el sistema va a procesar textos del sector salud, se necesita como mínimo: (1) un modelo NER biomédico que detecte fármacos, diagnósticos y procedimientos, (2) anonimización tanto de datos personales como de datos clínicos antes de enviar a un LLM externo, y (3) registro del tratamiento conforme a lo que exige la SIC (Superintendencia de Industria y Comercio) como autoridad de protección de datos en Colombia. Sin estas capas, cualquier uso de IA sobre textos clínicos es un riesgo legal directo.


<a id="sec-ejercicio-6"></a>
## 9. Ejercicio 6 – NER aplicado a documentos legales

En el dominio legal, los textos contienen:

- Nombres de personas y empresas.
- Juzgados, ciudades, fechas, números de contrato.
- Referencias a artículos de leyes, resoluciones, sentencias.

Todo esto es relevante para **cumplimiento** y para sistemas que revisan contratos, demandas o políticas.

### 9.1. Texto de ejemplo legal

```python
texto_legal = (
    "El abogado Carlos Ramírez presentó la demanda ante el Juzgado Primero de Lima "
    "el 12 de mayo de 2026 en representación de Grupo Andes S.A.S"
)

resultado_legal = ner_es(texto_legal)
resultado_legal
```

### 9.2. Recursos legales en Hugging Face

- Modelo NER legal en español (ejemplo):  
  - `agomez302/nlp-dr-ner`  
  - Página: [https://huggingface.co/agomez302/nlp-dr-ner](https://huggingface.co/agomez302/nlp-dr-ner)
- Guía de fine-tuning de modelos de clasificación de tokens para datos legales:  
  - [https://huggingface.co/blog/bikashpatra/legal-data-token-classification-fine-tuning](https://huggingface.co/blog/bikashpatra/legal-data-token-classification-fine-tuning)

### 9.3. Actividad

- A partir del resultado de NER general:
  - ¿Qué entidades son importantes para un flujo de revisión legal?  
  - ¿Qué entidades legales relevantes no aparecen (ej. tipo de proceso, número de expediente, referencia a norma)?  
- Discutan y registren entre ustedes qué necesitaría un sistema NER “serio” para legal:
  - Etiquetas más específicas.  
  - Entrenamiento con datasets legales.  
  - Reglas de negocio encima del modelo.

In [ ]:
# 9.1. NER general aplicado a texto legal
texto_legal = (
    "El abogado Carlos Ramírez presentó la demanda ante el Juzgado Primero de Lima "
    "el 12 de mayo de 2026 en representación de Grupo Andes S.A.S"
)

print("Texto:", texto_legal)
print("\nEntidades detectadas con modelo general:")
resultado_legal = ner_es(texto_legal)
for ent in resultado_legal:
    print(f"  -> {ent['word']} | {ent['entity_group']} | score={ent['score']:.3f}")


In [ ]:
# 9.2. Anonimización del texto legal
texto_legal_anon = anonimizar_entidades(texto_legal, resultado_legal)

print("Texto original:")
print(texto_legal)
print("\nTexto anonimizado:")
print(texto_legal_anon)
print("\n--- Análisis ---")
print("Entidades detectadas:", [ent['entity_group'] for ent in resultado_legal])
print("Datos NO detectados por el modelo general:")
print("  - Fecha: '12 de mayo de 2026' (no es entidad NER clásica)")
print("  - Tipo de proceso: 'demanda' (concepto legal, no nombre propio)")
print("  - Número de expediente: no aparece en el texto pero sería invisible para NER")


### 9.3. Respuestas del Ejercicio 6

**1. ¿Qué entidades son importantes para un flujo de revisión legal?**

En un documento legal, prácticamente todo es sensible: los nombres de las partes (demandante, demandado, abogado), el juzgado o tribunal, la ciudad, la empresa involucrada y las fechas. El modelo general detecta correctamente a Carlos Ramírez (PER), Lima o Juzgado Primero (LOC/ORG) y Grupo Andes S.A.S (ORG). Pero esto es solo la superficie del documento.

**2. ¿Qué entidades legales relevantes no aparecen?**

El modelo no detecta la fecha "12 de mayo de 2026", el tipo de proceso ("demanda"), referencias a artículos de ley, números de expediente, ni el rol profesional ("abogado"). En un sistema real de revisión legal, todos estos datos son fundamentales: un número de expediente identifica un caso único, una fecha determina plazos procesales, y la referencia a una norma indica qué ley se está aplicando. Nada de esto existe para un modelo NER general.

**3. Conexión con la legislación colombiana**

En Colombia, los documentos legales están protegidos por varias normas. La **Ley 1581 de 2012** aplica a cualquier dato personal contenido en demandas, contratos o expedientes judiciales. Además, la **Ley 1564 de 2012** (Código General del Proceso, Art. 123) establece que los expedientes judiciales tienen carácter reservado para terceros no vinculados al proceso. La **Ley 1712 de 2014** (Ley de Transparencia) también define excepciones de acceso cuando se trata de información que pueda afectar derechos de las personas.

Esto significa que si un sistema de IA procesa textos legales colombianos, no solo debe proteger nombres y empresas, sino también números de radicado, datos del proceso y cualquier información que permita identificar a las partes. Un modelo NER general deja expuesta la mayoría de esta información.

**4. ¿Qué necesitaría un sistema NER serio para el dominio legal?**

Necesitaría al menos tres cosas: (1) etiquetas más específicas como FECHA, EXPEDIENTE, NORMA, JUZGADO, ROL_PROFESIONAL, además de las clásicas PER/ORG/LOC; (2) entrenamiento con datasets de textos legales reales (demandas, contratos, sentencias colombianas); y (3) reglas de negocio adicionales para capturar patrones como números de radicado (formato específico de la Rama Judicial colombiana), referencias a artículos de ley, y plazos procesales. Sin estas capas, el modelo solo ve los nombres propios y deja pasar todo el contexto legal sensible.


<a id="sec-ejercicio-7"></a>
## 10. Ejercicio 7 – Selección autónoma de modelo y mini-experimento

En este punto, cada grupo debe:

1. Elegir un modelo NER diferente en español desde el catálogo:  
   - [https://huggingface.co/models?pipeline_tag=token-classification&language=es&search=ner](https://huggingface.co/models?pipeline_tag=token-classification&language=es&search=ner)
2. Justificar su elección según:
   - Idioma y dominio (general, biomédico, legal, etc.).  
   - Tamaño del modelo y viabilidad práctica.  
   - Documentación disponible.
   - fecha de publicación del modelo o actulización registrada

### 10.1. Plantilla de código

sugerencia de código:

```python
from transformers import pipeline

# Reemplaza esta línea con el modelo que hayan elijido
modelo_estudiante = "MMG/xlm-roberta-large-ner-spanish"

ner_est = pipeline(
    "ner",
    model=modelo_estudiante,
    aggregation_strategy="simple"
)

texto_prueba = (
    "Juan Pérez autorizó el tratamiento en la Clínica del Norte de Medellín "
    "y firmó el consentimiento informado."
)

ner_est(texto_prueba)
```

### 10.2. Preguntas de análisis

- ¿Qué etiquetas (`entity_group`) devuelve el modelo?  
- ¿Coinciden con las que esperaban por el dominio (general, clínico, legal, etc.)?  
- ¿Cómo cambiarían la estrategia de anonimización dependiendo del tipo de entidades detectadas?

In [ ]:
# 10.1. Selección de modelo: PlanTL-GOB-ES/bsc-bio-ehr-es-cantemist
# Modelo NER biomédico entrenado para detectar morfología de neoplasias (tumores)
from transformers import pipeline

modelo_estudiante = "PlanTL-GOB-ES/bsc-bio-ehr-es-cantemist"

ner_cantemist = pipeline(
    "ner",
    model=modelo_estudiante,
    aggregation_strategy="simple"
)

print(f"Modelo cargado: {modelo_estudiante}")
print("Dominio: Biomédico / Oncología clínica")
print("Entidades que detecta: MORFOLOGIA_NEOPLASIA (tipos de tumores y menciones oncológicas)")


In [ ]:
# 10.2. Texto de ejemplo: caso clínico oncológico
texto_clinico = (
    "Paciente femenina de 54 años con diagnóstico de carcinoma ductal infiltrante "
    "de mama derecha, receptor de estrógenos positivo. Se inició tratamiento con "
    "tamoxifeno 20 mg diarios. En control posterior se detectó metástasis pulmonar "
    "y adenocarcinoma de pulmón con mutación EGFR positiva."
)

print("=== Texto clínico ===")
print(texto_clinico)

# Resultado con modelo CANTEMIST (oncológico)
print("\n=== Entidades detectadas con modelo CANTEMIST (oncológico) ===")
resultado_cantemist = ner_cantemist(texto_clinico)
for ent in resultado_cantemist:
    print(f"  -> {ent['word']} | {ent['entity_group']} | score={ent['score']:.3f}")

# Comparación con modelo general
print("\n=== Entidades detectadas con modelo GENERAL (MMG/xlm-roberta) ===")
resultado_general = ner_es(texto_clinico)
if resultado_general:
    for ent in resultado_general:
        print(f"  -> {ent['word']} | {ent['entity_group']} | score={ent['score']:.3f}")
else:
    print("  (No detectó ninguna entidad)")


### 10.2. Respuestas del Ejercicio 7

**Justificación de la elección del modelo**

Elegimos `PlanTL-GOB-ES/bsc-bio-ehr-es-cantemist` porque es un modelo entrenado específicamente con casos clínicos oncológicos en español (dataset CANTEMIST: 1301 reportes de casos oncológicos anotados por expertos). Es relevante para nuestro curso porque los datos de salud son la categoría más sensible bajo cualquier ley de protección de datos, y un diagnóstico de cáncer es probablemente el ejemplo más claro de dato que requiere máxima protección.

**1. ¿Qué etiquetas devuelve el modelo?**

El modelo devuelve la etiqueta `MORFOLOGIA_NEOPLASIA`, que identifica menciones de tipos de tumores y morfología oncológica. En nuestro texto detecta entidades como "carcinoma ductal infiltrante", "metástasis pulmonar" y "adenocarcinoma de pulmón". Son términos que el modelo general ignora por completo porque no son nombres propios.

**2. ¿Coinciden con lo esperado para el dominio clínico?**

Sí. El modelo está diseñado para un dominio muy específico (oncología) y hace exactamente lo que se espera: identifica los tipos de tumor y sus características morfológicas. No detecta medicamentos (como tamoxifeno) ni datos personales, porque no fue entrenado para eso. Cada modelo cubre su nicho.

**3. ¿Cómo cambiaría la estrategia de anonimización?**

Con el modelo general solo anonimizaríamos nombres y lugares, pero toda la información clínica oncológica pasaría sin protección. Con el modelo CANTEMIST podemos además detectar y proteger los diagnósticos oncológicos. En un sistema real para el sector salud colombiano, habría que combinar ambos modelos: el general para PER/ORG/LOC y el biomédico para datos clínicos. Según la Ley 1581 (Art. 5), un diagnóstico de cáncer es un dato sensible cuyo tratamiento está prohibido salvo consentimiento explícito o necesidad médica directa.


<a id="sec-ejercicio-8"></a>
## 11. Ejercicio 8 – Caso de estudio de cumplimiento

**Objetivo:**  
Diseñar un mini caso realista que conecte NER con cumplimiento normativo en un dominio elegido por cada grupo.

### 11.1. Dominios sugeridos

- **Salud**: historias clínicas, recetas, informes de laboratorio.
- **Legal**: contratos, demandas, resoluciones, expedientes.
- **Educación**: listados de estudiantes, notas, informes de desempeño.
- **Finanzas**: formularios, extractos, reclamaciones, reportes de riesgo.

### 11.2. Entregable sugerido

Para cada grupo, el caso debe incluir:

1. **Texto de ejemplo** del dominio (inventado o anonimizado para clase).  
2. **Modelo NER utilizado** y breve justificación.  
3. **Salida NER** (lista de entidades).  
4. **Texto anonimizado** usando una función tipo `anonimizar_entidades`.  
5. **Riesgos identificados**: qué datos personales o sensibles aparecen, cuáles podrían faltar.  
6. **Limitaciones del modelo**: tipos de entidades que no detecta o que clasifica mal.  
7. **Relación con GDPR o con la norma local** (Colombia, México, Perú):  
   - ¿Qué principios de tratamiento se ven implicados (minimización, seguridad, finalidad, etc.)?

El propósito de este ejercicio es sirvir como puente entre la parte técnica (modelos NER) y la práctica profesional (arquitecturas de cumplimiento con LLM).

In [ ]:
# 11.1. NER Parametrizable para tickets de soporte de software
# Caso: sistema de soporte con acceso a repositorios y datos de sistemas críticos

import re

class NERParametrico:
    """
    NER extendido que combina el modelo base con reglas parametrizables
    según el dominio del ticket (seguros, finanzas, salud, etc.)
    """
    def __init__(self, ner_pipeline, dominio="general"):
        self.ner = ner_pipeline
        self.dominio = dominio
        # Patrones por dominio
        self.patrones = {
            "general": {
                "EMAIL": r"[\w.-]+@[\w.-]+\.\w+",
                "TELEFONO": r"\+?\d[\d\s-]{7,15}",
                "IP": r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}",
            },
            "finanzas": {
                "CUENTA_BANCARIA": r"\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}",
                "NIT": r"\d{9,10}-\d",
            },
            "salud": {
                "HISTORIA_CLINICA": r"HC[\s-]?\d{5,10}",
                "CEDULA": r"C\.?C\.?\s*\d{6,10}",
            },
            "seguros": {
                "POLIZA": r"POL[\s-]?\d{6,12}",
                "SINIESTRO": r"SIN[\s-]?\d{6,10}",
            },
        }

    def detectar(self, texto):
        # 1. Entidades del modelo NER base
        entidades = self.ner(texto)
        resultado = [(ent["word"], ent["entity_group"], ent["start"], ent["end"]) for ent in entidades]

        # 2. Agregar patrones generales + los del dominio específico
        dominios_a_buscar = ["general"]
        if self.dominio != "general":
            dominios_a_buscar.append(self.dominio)

        for dom in dominios_a_buscar:
            if dom in self.patrones:
                for etiqueta, patron in self.patrones[dom].items():
                    for match in re.finditer(patron, texto):
                        resultado.append((match.group(), etiqueta, match.start(), match.end()))

        return sorted(resultado, key=lambda x: x[2])

    def anonimizar(self, texto):
        entidades = self.detectar(texto)
        # Reemplazar de atrás hacia adelante
        ents_sorted = sorted(entidades, key=lambda x: x[2], reverse=True)
        texto_anon = texto
        for word, group, start, end in ents_sorted:
            texto_anon = texto_anon[:start] + f"[{group}]" + texto_anon[end:]
        return texto_anon

print("Clase NERParametrico definida correctamente.")
print("Dominios disponibles: general, finanzas, salud, seguros")


In [ ]:
# 11.2. Caso práctico: Ticket de soporte con datos de múltiples dominios

ticket_soporte = (
    "Ticket #4521 - Prioridad Alta\n"
    "Agente: Carlos Ramírez (carlos.ramirez@bsginstitute.com)\n"
    "Cliente: Laura Medina, CC 52.489.331, póliza POL-20258741\n"
    "Sistema afectado: Módulo de reclamaciones (servidor 192.168.1.45)\n"
    "Descripción: La cliente reporta que su siniestro SIN-887421 no refleja \n"
    "el pago de la reclamación por consulta en Clínica San Rafael de Bogotá. \n"
    "Historia clínica HC-74829 adjunta. Cuenta destino: 4512-7823-0091-3345."
)

print("=== TICKET DE SOPORTE ORIGINAL ===")
print(ticket_soporte)

# Usar NER parametrizado para dominio de seguros
ner_seguros = NERParametrico(ner_es, dominio="seguros")

print("\n=== ENTIDADES DETECTADAS (dominio: seguros) ===")
entidades = ner_seguros.detectar(ticket_soporte)
for word, group, start, end in entidades:
    print(f"  -> {word:30s} | {group}")

print("\n=== TICKET ANONIMIZADO ===")
ticket_anon = ner_seguros.anonimizar(ticket_soporte)
print(ticket_anon)


In [ ]:
# 11.3. Mismo ticket, diferente dominio parametrizado
# Si cambiamos el dominio a "salud", detecta historia clínica y cédula
# Si usamos "finanzas", detecta la cuenta bancaria y NIT

print("=== Comparación de detección por dominio ===\n")

for dominio in ["general", "seguros", "salud", "finanzas"]:
    ner_param = NERParametrico(ner_es, dominio=dominio)
    ents = ner_param.detectar(ticket_soporte)
    extras = [f"{w} ({g})" for w, g, s, e in ents if g not in ("PER", "ORG", "LOC", "MISC")]
    print(f"Dominio [{dominio:10s}]: {len(ents)} entidades totales")
    if extras:
        print(f"  Datos extra detectados: {", ".join(extras)}")
    print()


### 11.2. Respuestas del Ejercicio 8

**Caso de estudio: Ticket de soporte de software con datos críticos**

El escenario es un sistema de tickets que tiene acceso a repositorios de código y a datos confidenciales de clientes en dominios como seguros, finanzas y salud. Un solo ticket puede contener nombres, correos, IPs de servidores, números de póliza, cuentas bancarias e historias clínicas, todo mezclado.

**1. ¿Por qué un NER parametrizable?**

Porque el modelo NER base solo detecta nombres, lugares y organizaciones. No ve correos, IPs, pólizas ni cuentas bancarias. La solución es agregarle reglas (expresiones regulares) que se activan según el dominio del ticket. Si el ticket es de seguros, se buscan pólizas y siniestros. Si es de salud, se buscan historias clínicas. Así el mismo sistema se adapta sin necesitar un modelo diferente para cada caso.

**2. ¿Qué riesgos tiene este tipo de ticket?**

Un ticket como este concentra datos de varios dominios sensibles a la vez: datos personales (nombre, cédula, correo), datos financieros (cuenta bancaria), datos de salud (historia clínica) y datos de infraestructura (IP del servidor). Si este ticket se envía a un LLM externo para generar una respuesta automática, se estarían filtrando datos protegidos por la Ley 1581 de 2012 en varias categorías simultáneamente.

**3. Conexión con cumplimiento colombiano**

Bajo la Ley 1581, este ticket viola varios principios si no se anonimiza antes de procesarse con IA externa: el principio de **seguridad** (Art. 4 lit. g) porque no se protegen los datos con medidas adecuadas, el de **confidencialidad** (Art. 4 lit. h) porque se exponen datos a terceros no autorizados, y el de **acceso y circulación restringida** (Art. 4 lit. f) porque datos sensibles de salud circulan fuera del contexto médico autorizado. La SIC podría sancionar a la empresa por no implementar controles técnicos antes de compartir esta información.

**4. Ventaja del enfoque parametrizable**

La clase `NERParametrico` muestra que no se necesita un modelo diferente para cada dominio. Se puede usar el mismo modelo base (que detecta PER/ORG/LOC) y agregarle capas de reglas según el contexto. Esto es más práctico y económico que entrenar modelos separados, y permite que un equipo de cumplimiento ajuste las reglas sin tocar el modelo de IA.


<a id="sec-recursos"></a>
## 12. Recursos y datasets recomendados

### 12.1. NER general en español

- Modelo `MMG/xlm-roberta-large-ner-spanish` (CoNLL-2002 ES):  
  - Página del modelo: https://huggingface.co/MMG/xlm-roberta-large-ner-spanish
- Introducción a token classification / NER en la Hugging Face course:  
  - https://huggingface.co/learn/llm-course/chapter7/2

### 12.2. Salud / medicina

- Repositorio de modelos biomédicos en español (PlanTL-GOB-ES):  
  - https://github.com/PlanTL-GOB-ES/lm-biomedical-clinical-es
- Datasets en Hugging Face:  
  - CANTEMIST NER: https://huggingface.co/datasets/PlanTL-GOB-ES/cantemist-ner
  - PharmaCoNER: https://huggingface.co/datasets/PlanTL-GOB-ES/pharmaconer

### 12.3. Legal

- Modelo NER legal en español:  
  - `agomez302/nlp-dr-ner`: https://huggingface.co/agomez302/nlp-dr-ner
- Guía de fine-tuning para clasificación de tokens en datos legales:  
  - https://huggingface.co/blog/bikashpatra/legal-data-token-classification-fine-tuning

### 12.4. Marco regulatorio

- GDPR – información general y guías:
  - https://gdpr.eu/what-is-gdpr/  
  - Ejemplos de contenidos para formación en GDPR: https://usercentrics.com/knowledge-hub/gdpr-training/
- Colombia – Ley 1581 de 2012:
  - Texto oficial: https://www.alcaldiabogota.gov.co/sisjur/normas/Norma1.jsp?i=49981  
- México – LFPDPPP: 
  - Guía de cumplimiento: https://resguard-solutions.com/blog/en/mexico-lfpdppp-data-protection-guide/  
- Perú – Ley 29733:
  - Guía de cumplimiento: https://resguard-solutions.com/blog/en/peru-law-29733-data-protection-guide/

<a id="sec-cierre"></a>
## 13. Cierre y reflexión final

En este notebook hemos visto cómo:

- Un modelo NER en español puede detectar entidades como personas, organizaciones y lugares en textos reales.
- Esa detección se puede usar para **anonimizar** textos antes de enviarlos a un LLM o a otra API, apoyando principios de **minimización** y **seguridad** recogidos en GDPR y en las leyes locales de Colombia, México y Perú.
- Los modelos generales tienen límites importantes, especialmente en dominios sensibles como salud y legal, donde suelen ser necesarios modelos NER **especializados** entrenados con datasets clínicos o jurídicos.
- Además de NER, usamos la tarea de **rellenar máscaras** (`fill-mask`) para observar cómo un modelo de lenguaje completa frases con huecos y cómo esto puede revelar **sesgos estadísticos** o hacer que ciertos datos enmascarados sean demasiado fáciles de adivinar.

La lección central es que la técnica por sí sola no garantiza cumplimiento:

- NER y `fill-mask` son **herramientas** dentro de un sistema más amplio que debe incluir políticas internas, formación, roles claros (como el DPO), contratos adecuados con proveedores y auditorías periódicas.
- Desde la perspectiva de diseño de aplicaciones de IA con LLM, el desafío está en combinar modelos, reglas de negocio y conocimiento legal para construir sistemas que sean útiles, pero también **seguros, no discriminatorios y alineados con la normativa vigente**.

Este es un Notebook en desarrollo y estos serían los temas a cubrir en la próxima sesión:

- Evaluar cuantitativamente la calidad de distintos modelos NER (precisión, recall, F1).  
- Integrar NER como paso previo en un pipeline RAG que filtre y anonimice documentos antes de indexarlos.  
- Usar sistemáticamente tareas tipo `fill-mask` para medir sesgos y riesgos de re-identificación en modelos de lenguaje.